# 04 — Deep Models

Train a **Dense Autoencoder** on tabular snapshots and an **LSTM Autoencoder**
on sliding windows for each dataset. Anomaly score = reconstruction MSE.


In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import set_seed, PROCESSED_DIR, METRICS_DIR, save_figure
from src.models.autoencoder import DenseAutoencoderAD
from src.models.lstm_autoencoder import LSTMAutoencoderAD
from src.evaluation import (
    best_f1_threshold, compute_metrics, plot_roc, plot_pr, save_metrics_row,
)

set_seed(42)
sns.set_theme(style='whitegrid', context='notebook')


In [ ]:
def load_tabular(name):
    d = np.load(PROCESSED_DIR / f'{name}_tabular.npz')
    return d['X_train'], d['X_val'], d['X_test'], d['y_test']

def load_windows(name):
    d = np.load(PROCESSED_DIR / f'{name}_windows.npz')
    return d['X_train'], d['X_val'], d['X_test'], d['y_test']

# Cap training sample sizes for tractable runtimes on CPU.
CAP_TAB = 150_000
CAP_WIN = 30_000

def cap(X, n, seed=0):
    if len(X) <= n: return X
    rng = np.random.default_rng(seed)
    return X[rng.choice(len(X), n, replace=False)]


## Dense Autoencoder

In [ ]:
rows = []
for key, label in [('hai', 'HAI 21.03'), ('morris', 'Morris gas')]:
    X_tr, X_val, X_te, y_te = load_tabular(key)
    X_tr = cap(X_tr, CAP_TAB)
    ae = DenseAutoencoderAD(input_dim=X_tr.shape[1], epochs=25, batch_size=512)
    print(f'— DenseAE on {label} — train n={len(X_tr):,}')
    ae.fit(X_tr, X_val)
    scores = ae.score(X_te)

    rng = np.random.default_rng(7)
    tune_idx = np.unique(rng.choice(len(y_te), min(5_000, len(y_te)), replace=False))
    thr, _ = best_f1_threshold(scores[tune_idx], y_te[tune_idx])

    m = compute_metrics(scores, y_te, thr, 'DenseAE', key)
    print(f'  F1={m.f1:.3f}  ROC-AUC={m.roc_auc:.3f}  PR-AUC={m.pr_auc:.3f}')
    save_metrics_row(m, METRICS_DIR)
    rows.append((label, ae, scores, y_te))


In [ ]:
# Plot training loss curves.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, (label, ae, _, _) in zip(axes, rows):
    ax.plot(ae.history.history['loss'], label='train')
    if 'val_loss' in ae.history.history:
        ax.plot(ae.history.history['val_loss'], label='val')
    ax.set_title(f'DenseAE loss — {label}')
    ax.set_xlabel('epoch'); ax.set_ylabel('MSE'); ax.legend()
plt.tight_layout()
save_figure(fig, 'dense_ae_loss', subdir='04_deep')
plt.show()


## LSTM Autoencoder

In [ ]:
lstm_rows = []
for key, label in [('hai', 'HAI 21.03'), ('morris', 'Morris gas')]:
    X_tr, X_val, X_te, y_te = load_windows(key)
    X_tr = cap(X_tr, CAP_WIN)
    window, n_feat = X_tr.shape[1], X_tr.shape[2]
    lstm = LSTMAutoencoderAD(window=window, n_features=n_feat, epochs=10, batch_size=256)
    print(f'— LSTM-AE on {label} — train n={len(X_tr):,}, window={window}')
    lstm.fit(X_tr, X_val[:5000])
    scores = lstm.score(X_te)

    rng = np.random.default_rng(7)
    tune_idx = np.unique(rng.choice(len(y_te), min(2_000, len(y_te)), replace=False))
    thr, _ = best_f1_threshold(scores[tune_idx], y_te[tune_idx])

    m = compute_metrics(scores, y_te, thr, 'LSTM_AE', key)
    print(f'  F1={m.f1:.3f}  ROC-AUC={m.roc_auc:.3f}  PR-AUC={m.pr_auc:.3f}')
    save_metrics_row(m, METRICS_DIR)
    lstm_rows.append((label, lstm, scores, y_te))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, (label, lstm, _, _) in zip(axes, lstm_rows):
    ax.plot(lstm.history.history['loss'], label='train')
    if 'val_loss' in lstm.history.history:
        ax.plot(lstm.history.history['val_loss'], label='val')
    ax.set_title(f'LSTM-AE loss — {label}')
    ax.set_xlabel('epoch'); ax.set_ylabel('MSE'); ax.legend()
plt.tight_layout()
save_figure(fig, 'lstm_ae_loss', subdir='04_deep')
plt.show()


In [ ]:
# ROC and PR curves for deep models.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, _, scores, y in rows:
    plot_roc(axes[0], y, scores, f'DenseAE / {label}')
    plot_pr (axes[1], y, scores, f'DenseAE / {label}')
for label, _, scores, y in lstm_rows:
    plot_roc(axes[0], y, scores, f'LSTM-AE / {label}')
    plot_pr (axes[1], y, scores, f'LSTM-AE / {label}')
axes[0].plot([0,1],[0,1],'--',color='gray',linewidth=0.8)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('ROC — deep models')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].set_title('PR — deep models')
for ax in axes: ax.legend(fontsize=8)
plt.tight_layout()
save_figure(fig, 'deep_roc_pr', subdir='04_deep')
plt.show()
